In [1]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer

In [2]:
model_name = 'distilbert-base-uncased'
lr = 0.00001
epochs = 5
wd = 0.005

In [19]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
training_args = TrainingArguments(output_dir='out/', report_to='none', learning_rate=lr,
                                  num_train_epochs=epochs, weight_decay=wd,
                                  #eval_strategy='epoch',
                                  save_strategy='no')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
class YelpDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [7]:
data = pd.read_csv('data/yelp.txt', sep='\t')
data

,Sentence,Style
0,ever since joes has changed hands it's just go...,negative
1,there is definitely not enough room in that pa...,negative
2,so basically tasted watered down.,negative
3,she said she'd be back and disappeared for a f...,negative
4,i can't believe how inconsiderate this pharmac...,negative
...,...,...
1995,"this is a fantastic location, especially becau...",positive
1996,i was pleased with their service.,positive
1997,you get so much for the price and the food is ...,positive
1998,they are so fresh and yummy.,positive


In [8]:
encodings = tokenizer(data['Sentence'].values.tolist())

In [11]:
encodings

{'input_ids': [[101, 2412, 2144, 3533, 2015, 2038, 2904, 2398, 2009, 1005, 1055, 2074, 5407, 4788, 1998, 4788, 1012, 102], [101, 2045, 2003, 5791, 2025, 2438, 2282, 1999, 2008, 2112, 1997, 1996, 6891, 1012, 102], [101, 2061, 10468, 12595, 27129, 2091, 1012, 102], [101, 2016, 2056, 2016, 1005, 1040, 2022, 2067, 1998, 5419, 2005, 1037, 2261, 2781, 1012, 102], [101, 1045, 2064, 1005, 1056, 2903, 2129, 4297, 5644, 18688, 3686, 2023, 13882, 2003, 1012, 102], [101, 2074, 2187, 1998, 2165, 2009, 2125, 1996, 3021, 1012, 102], [101, 2009, 3475, 1005, 1056, 6659, 1010, 2021, 2009, 3475, 1005, 1056, 2200, 2204, 2593, 1012, 102], [101, 5791, 9364, 2008, 1045, 2071, 2025, 2224, 2026, 5798, 5592, 999, 102], [101, 2047, 3954, 1010, 1045, 2657, 1011, 2021, 1045, 2123, 1005, 1056, 2113, 1996, 4751, 1012, 102], [101, 2021, 2009, 2763, 19237, 2205, 999, 102], [101, 2057, 4133, 2091, 1998, 2057, 2288, 2070, 2428, 4030, 1998, 13971, 2326, 1012, 102], [101, 1996, 3715, 2106, 2421, 28616, 2080, 11350, 1998, 

In [14]:
labels = [1 if c=='positive' else 0 for c in data['Style'].values.tolist()]
labels

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,


In [16]:
dataset = YelpDataset(encodings=encodings, labels=labels)
dataset

In [17]:
next(iter(dataset))

{'input_ids': tensor([ 101, 2412, 2144, 3533, 2015, 2038, 2904, 2398, 2009, 1005, 1055, 2074,
         5407, 4788, 1998, 4788, 1012,  102]),
 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
 'labels': tensor(0)}

In [20]:
trainer = Trainer(model=model, args=training_args,
                  train_dataset=dataset, #eval_dataset=...,
                  processing_class=tokenizer)

In [21]:
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,0.348927
1000,0.149480


TrainOutput(global_step=1250, training_loss=0.22032303771972656, metrics={'train_runtime': 1012.1679, 'train_samples_per_second': 9.88, 'train_steps_per_second': 1.235, 'total_flos': 45413549904864.0, 'train_loss': 0.22032303771972656, 'epoch': 5.0})

In [ ]:
trainer.predict(..)